# DataHub — Galaxy Star Formation Rate Prediction
## Midterm + Final Scientific Report Template

This notebook is both your **analysis code** and your **scientific report**. Use the **same notebook for both submission stages**.

- **Part I — Midterm Report:** complete all required Part I sections and submit the notebook.
- **Part II — Final Report:** continue in the same notebook, preserve Part I, and complete the final analysis, code, results, and discussion.

## Participant Information

**Application ID**

**email:**

---
# Part I — Midterm Report

**Goal:** establish a scientifically motivated modeling plan. Part I should demonstrate that you understand the prediction problem, have inspected the data carefully, established a defensible validation procedure, and produced preliminary results.

### Part I deliverables
1. Problem & objective
2. Data loading and quality checks
3. Exploratory data analysis
4. Data preparation and feature engineering
5. Validation and evaluation strategy
6. Baseline results
7. Preliminary model / proposed method
8. Midterm summary and next-step plan

## 1. Problem & Objective — **STUDENT RESPONSE**

In approximately **150–250 words**, explain:
- what quantity you are predicting;
- what information is available as input;
- why this is a non-trivial regression problem;
- one or more astrophysical considerations that may affect the prediction.

**Response:**  
...

## 2. Setup, Reproducibility & Data Loading

The fixed cells below establish common imports, a random seed, and basic schema checks. You may change the data path if necessary, but do not silently alter the raw data.

In [1]:
# DO NOT MODIFY — core imports and seed
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 100)
print("Random seed:", RANDOM_SEED)

Random seed: 42


In [2]:
# You may change DATA_PATH only if needed.
CANDIDATES = [
    Path("../../Galaxy Features with Position and Velocity for Star Formation Rate Prediction.csv"),  # work/팀원/ 에서 실행할 때
    Path("Galaxy Features with Position and Velocity for Star Formation Rate Prediction.csv"),
    Path("/mnt/data/Galaxy Features with Position and Velocity for Star Formation Rate Prediction.csv"),
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Dataset not found. Place the course CSV beside this notebook or update DATA_PATH.")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

Shape: (208437, 15)


,subhalo_id,log10_halo_mass,log10_stellar_mass,log10_BH_mass,log10_main_prog_mass,log10_main_prog_stellar_mass,log10_mass_ratio,log10_overdensity_plus_1,pos_x,pos_y,pos_z,vel_x,vel_y,vel_z,sfr_linear
0,0.0,14.807566,12.812004,10.443699,14.631294,12.589542,-3.239739,2.383863,21.290455,101.703239,118.123352,-287.557892,-62.394615,-30.580877,107.896912
1,1.0,13.870680,12.091579,8.625058,13.819390,11.916792,-6.000000,2.025288,20.735514,101.805397,120.937294,786.127563,258.246796,-680.667908,68.733315
2,2.0,13.672584,12.032370,9.829654,13.620131,11.864630,-6.000000,1.704644,18.711687,103.225121,120.417160,890.943054,-756.617004,-358.727631,0.232002
3,3.0,13.650022,12.019966,9.312719,13.577394,11.849984,-6.000000,1.833376,23.493862,102.198166,113.814438,-92.127396,532.597473,544.153687,12.698732
4,4.0,13.131606,11.563723,9.452775,13.026983,11.395634,-6.000000,1.830341,24.067015,101.780312,112.495789,-187.708572,-224.876968,978.109558,0.715540


In [3]:
# DO NOT MODIFY — schema checks
TARGET = "sfr_linear"
ID_COLUMN = "subhalo_id"
EXPECTED_COLUMNS = [
    "subhalo_id", "log10_halo_mass", "log10_stellar_mass", "log10_BH_mass",
    "log10_main_prog_mass", "log10_main_prog_stellar_mass", "log10_mass_ratio",
    "log10_overdensity_plus_1", "pos_x", "pos_y", "pos_z",
    "vel_x", "vel_y", "vel_z", "sfr_linear"
]
missing = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
if missing:
    raise ValueError(f"Missing expected columns: {missing}")
n_missing_target = int(df[TARGET].isna().sum())
print(f"Missing target values: {n_missing_target}")
if n_missing_target > 0:
    print(f"Dropping {n_missing_target} row(s) with missing target before modeling.")
    df = df.dropna(subset=[TARGET]).copy()
print(f"Schema check passed. Samples available for modeling: {len(df):,}")


Missing target values: 0
Schema check passed. Samples available for modeling: 208,437


### 2.1 Data Quality Notes — **STUDENT RESPONSE**

Report the dataset size, missing values, duplicate handling (if any), and any other data-quality issue you found. Remember that **`sfr_linear = 0` is a valid physical target and is not the same as a missing value**.

**Response:**  
...

## 3. Exploratory Data Analysis (EDA) — **STUDENT WORK**

Your EDA should be selective and scientifically useful rather than a gallery of plots. Include **at least two analyses/figures that influence a later modeling decision**. At minimum, examine the target distribution and relevant relationships between predictors and SFR.

For every important figure/table, add a short interpretation directly below it: **What does it show? Why might the pattern occur? How does it affect your modeling choices?**

In [4]:
# STUDENT TODO — starter summary
print(df[TARGET].describe(percentiles=[0.5, 0.9, 0.99]))
zero_fraction = (df[TARGET] == 0).mean()
print(f"Exact zero-SFR fraction: {zero_fraction:.2%}")

# Add your EDA here.

# FINAL_FEATURES는 4절에서 만들어지므로, 3절에서는 같은 규칙으로 피처 목록을 직접 만든다.
FEATURES = [c for c in df.columns if c not in [ID_COLUMN, TARGET]]

# 1. 데이터 크기
print("shape:", df.shape)

# 2. 결측치와 중복
print("NaN 총개수:", int(df.isna().sum().sum()))
print(
    "전체 열 중복 행:", int(df.duplicated().sum()),
    "| subhalo_id 중복:",
    int(df[ID_COLUMN].duplicated().sum())
)

# 3. SFR이 정확히 0인 은하
print(
    f"SFR == 0 비율: {(df[TARGET] == 0).mean():.2%} "
    f"({int((df[TARGET] == 0).sum())}개)"
)

# 4. SFR의 분포
print(df[TARGET].describe(
    percentiles=[0.5, 0.9, 0.99, 0.999]
).round(3))

# 5. 센티널 값의 개수
sentinels = {
    "BH <= 5.01": (df["log10_BH_mass"] <= 5.01).sum(),
    "mass_ratio == -6": (df["log10_mass_ratio"] == -6).sum(),
    "main_prog_mass == 6.0": (
        df["log10_main_prog_mass"] == 6.0
    ).sum(),
}

for k, v in sentinels.items():
    print(f"{k}: {v:,}개 ({v / len(df):.1%})")

# 6. 각 피처의 범위
print("피처 범위:")
print(df[FEATURES].agg(
    ["min", "median", "max"]
).T.round(2))


count    208437.000000
mean          3.709985
std          11.794583
min           0.000000
50%           1.631928
90%           7.554342
99%          34.859419
max        2145.372800
Name: sfr_linear, dtype: float64
Exact zero-SFR fraction: 8.66%
shape: (208437, 15)
NaN 총개수: 0


전체 열 중복 행: 0 | subhalo_id 중복: 0
SFR == 0 비율: 8.66% (18048개)
count    208437.000
mean          3.710
std          11.795
min           0.000
50%           1.632
90%           7.554
99%          34.859
99.9%       112.781
max        2145.373
Name: sfr_linear, dtype: float64
BH <= 5.01: 121,546개 (58.3%)
mass_ratio == -6: 175,778개 (84.3%)
main_prog_mass == 6.0: 762개 (0.4%)
피처 범위:
                                  min  median      max
log10_halo_mass                  8.00   11.38    14.83
log10_stellar_mass               9.30    9.84    12.96
log10_BH_mass                    5.00    5.00    11.16
log10_main_prog_mass             6.00   11.24    14.70
log10_main_prog_stellar_mass     6.00    9.63    12.75
log10_mass_ratio                -6.00   -6.00    -0.00
log10_overdensity_plus_1        -0.58    0.66     2.51
pos_x                            0.00   91.63   200.00
pos_y                            0.00  106.82   200.00
pos_z                            0.00  101.47   200.00
vel_x           

### 3.1 EDA Findings — **STUDENT RESPONSE**

**Finding 1:** ...  
**Finding 2:** ...  
**Additional finding(s):** ...  

**Modeling decisions motivated by the EDA:** ...

## 4. Data Preparation & Feature Engineering — **STUDENT WORK**

Document all choices that affect model inputs. Discuss feature inclusion/exclusion, missing-value treatment, transformations, scaling, and engineered features where applicable. **`subhalo_id` must not be used as a predictive feature.**

In [5]:
# Default allowed feature set. You may modify FINAL_FEATURES, but never include subhalo_id or sfr_linear.
ALLOWED_FEATURES = [c for c in df.columns if c not in [ID_COLUMN, TARGET]]
FINAL_FEATURES = ALLOWED_FEATURES.copy()

assert ID_COLUMN not in FINAL_FEATURES
assert TARGET not in FINAL_FEATURES
assert set(FINAL_FEATURES).issubset(set(ALLOWED_FEATURES))

X = df[FINAL_FEATURES].copy()
y = df[TARGET].copy()
print(f"Using {len(FINAL_FEATURES)} features:")
print(FINAL_FEATURES)

# STUDENT TODO: add preprocessing / feature engineering if desired.

Using 13 features:
['log10_halo_mass', 'log10_stellar_mass', 'log10_BH_mass', 'log10_main_prog_mass', 'log10_main_prog_stellar_mass', 'log10_mass_ratio', 'log10_overdensity_plus_1', 'pos_x', 'pos_y', 'pos_z', 'vel_x', 'vel_y', 'vel_z']


### 4.1 Preprocessing Rationale — **STUDENT RESPONSE**

| Choice | What you did | Why |
|---|---|---|
| Feature selection |  |  |
| Missing values |  |  |
| Scaling / transformation |  |  |
| Feature engineering |  |  |

**Additional explanation:** ...